# 📱 Impacto das Redes Sociais na Saúde e no Desempenho de Estudantes
### Relatório de Análise — Regressão Linear
*Big Data e Processamento Otimizado com Python II*

| | |
|---|---|
| **Autora** | Indiara, Dialuan e Guilherme |
| **Base de dados** | `estudantes.parquet` — 500.000 estudantes, 48 variáveis |
| **Ferramentas** | Python, Polars, scikit-learn, Matplotlib e Seaborn |
| **Continuação de** | Parte 1 (faixas), Parte 2 (distribuição) e Correlações |
| **Esta etapa** | Regressão linear simples e múltipla, com validação treino/teste |

**Pergunta de pesquisa:** quanto o uso de redes sociais ajuda a explicar a **saúde** (bem-estar e sono) e o **desempenho** (frequência e atenção)?

🗺️ **Roteiro do estudo:** Parte 1 → Parte 2 → Correlações → **▶ Regressão** → Conclusão integrada

---

## 1. Resumo executivo

| Modelo | O que explica | R² (teste) | Leitura |
|---|---|---|---|
| **C.** Sono ~ redes | 😴 Sono | **0,580** | Forte: redes explicam 58% da variação do sono |
| **B.** Bem-estar ~ redes + sono | 💚 Bem-estar | **0,511** | Forte: o sono pesa mais que as redes |
| **A.** Bem-estar ~ redes | 💚 Bem-estar | **0,402** | Forte: cada hora em redes se associa a **-5,2 pontos** |
| **E.** Atenção ~ redes + sono | 🎯 Atenção | **0,197** | Moderada: as redes pesam mais que o sono |
| **D.** Frequência ~ redes + sono | 🏫 Frequência | **0,079** | Baixa: a maior parte da variação vem de outros fatores |

> 💡 **Mensagem central:** o uso de redes sociais **explica bem a saúde** (40% a 58% da variação) e **pouco o desempenho** (8% a 20%). Quando o sono entra no modelo, ele é o fator dominante para o **bem-estar**, mas para **atenção e frequência** as redes continuam pesando mais.

---

## 2. Introdução e conexão com as etapas anteriores

| Etapa | O que mostrou |
|---|---|
| **Parte 1** | Bem-estar médio de **74,11** (pouco uso e sono bom) contra **45,15** (alto uso e sono baixo) |
| **Parte 2** | Uso médio de 3,36 h/dia, distribuição simétrica e faixa *Alto* (≥ 5 h) com cerca de 11% dos estudantes |
| **Correlações** | Redes × sono (r = -0,76), redes × bem-estar (-0,63), redes × atenção (-0,43), redes × frequência (-0,28) |
| **Esta etapa** | **Quantifica o efeito** de cada hora em redes e mede **quanto os modelos explicam** e **quão bem preveem** em dados novos |

As correlações indicam a **força** da associação. A regressão acrescenta duas informações: o **tamanho do efeito em unidades reais** (por exemplo, pontos de bem-estar por hora de uso) e a **comparação do peso** entre redes e sono quando os dois entram juntos.

---

## 3. Dados e metodologia

### 3.1 Modelos estimados

| Modelo | Variável resposta | Preditores |
|---|---|---|
| **A** | Bem-estar (`indice_bem_estar`) | Horas em redes |
| **B** | Bem-estar (`indice_bem_estar`) | Horas em redes + horas de sono |
| **C** | Sono (`horas_sono`) | Horas em redes |
| **D** | Frequência (`taxa_frequencia_aulas`) | Horas em redes + horas de sono |
| **E** | Atenção (`capacidade_atencao_minutos`) | Horas em redes + horas de sono |

Os modelos A e B permitem ver **o que muda no efeito das redes quando o sono entra na equação**.

### 3.2 Método e métricas

- **Regressão linear** (scikit-learn), com **separação treino/teste de 80/20** (`random_state = 42`). As métricas são calculadas no **conjunto de teste** (100.000 estudantes).
- **R²:** fração da variação explicada pelo modelo (de 0 a 1).
- **MAE:** erro absoluto médio, na unidade da variável resposta.
- **RMSE:** raiz do erro quadrático médio, que penaliza mais os erros grandes.
- **Coeficiente:** variação esperada na resposta para **+1 unidade** do preditor, mantendo os demais constantes.
- **Beta padronizado:** coeficiente com as variáveis padronizadas, que permite **comparar o peso** de preditores com unidades diferentes.

### 3.3 Preparação dos dados
Importa as bibliotecas, lê a base com `scan_parquet` e define as funções de apoio.

In [1]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

inicial = pl.scan_parquet("estudantes.parquet")
resultados = []

### 3.4 Função de ajuste
A função `ajustar()` padroniza todos os modelos: seleciona as colunas, remove ausentes, separa treino e teste, ajusta a regressão, calcula as métricas e gera **um gráfico por modelo**: a reta de regressão sobre os dados (um preditor) ou o gráfico real × previsto (dois preditores).

In [2]:
def ajustar(titulo, alvo, feats):
    """Ajusta a regressão (treino/teste 80/20), imprime as métricas e mostra o gráfico do modelo."""
    df = inicial.select(feats + [alvo]).drop_nulls().collect()
    X, y = df.select(feats).to_numpy(), df[alvo].to_numpy()
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

    modelo = LinearRegression().fit(Xtr, ytr)
    pred = modelo.predict(Xte)
    r2 = r2_score(yte, pred)
    mae = mean_absolute_error(yte, pred)
    rmse = np.sqrt(mean_squared_error(yte, pred))

    # Betas padronizados: permitem comparar o peso de cada preditor
    zX, zy = (X - X.mean(0)) / X.std(0), (y - y.mean()) / y.std()
    betas = LinearRegression().fit(zX, zy).coef_

    print(f"\n{'=' * 70}\n{titulo}\n{'=' * 70}")
    print(f"n = {df.height:,} | R² (teste) = {r2:.4f} | MAE = {mae:.3f} | RMSE = {rmse:.3f}")
    print(f"Intercepto: {modelo.intercept_:.3f}")
    for f, c, b in zip(feats, modelo.coef_, betas):
        print(f"  {f:35s} coef = {c:+.4f} | beta padronizado = {b:+.4f}")
    if len(feats) > 1:
        print("Correlação entre preditores (checar colinearidade):")
        display(df.select(feats).corr())

    # Gráfico de regressão (amostra de 3.000 pontos do teste)
    idx = np.random.default_rng(42).choice(len(yte), size=3000, replace=False)
    fig, ax = plt.subplots(figsize=(7, 4.5))
    if len(feats) == 1:
        ax.scatter(Xte[idx, 0], yte[idx], alpha=0.2, s=8)
        xs = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
        ax.plot(xs, modelo.predict(xs), color="red")
        ax.set(xlabel=feats[0], ylabel=alvo, title=f"{titulo}: dados e reta de regressão")
    else:
        ax.scatter(yte[idx], pred[idx], alpha=0.2, s=8)
        ax.plot([yte.min(), yte.max()], [yte.min(), yte.max()], color="red")
        ax.set(xlabel="Real", ylabel="Previsto", title=f"{titulo}: real × previsto")
    plt.tight_layout()
    plt.show()

    return {"modelo": titulo, "alvo": alvo, "preditores": ", ".join(feats),
            "n": df.height, "r2_teste": round(r2, 4), "mae": round(mae, 3), "rmse": round(rmse, 3)}

> 🔎 **Interpretação.** Usar uma única função garante que todos os modelos sejam avaliados com o **mesmo critério**, o que torna a comparação justa.
>
> ▶️ **Sobre os gráficos:** a versão simplificada gera apenas o gráfico de regressão de cada modelo, **sem o histograma de resíduos**. Os gráficos aparecem ao **executar o notebook**; as métricas exibidas abaixo vêm da execução anterior.

---
## 4. Resultados

### 4.1 Modelos de saúde
Bem-estar e sono como variáveis resposta.

#### Modelo A — Bem-estar ~ redes sociais

In [3]:
resultados.append(ajustar("A. Bem-estar ~ redes sociais", "indice_bem_estar", ["horas_redes_sociais"]))


A. Bem-estar ~ redes sociais
n = 500,000 | R² (teste) = 0.4021 | MAE = 6.535 | RMSE = 8.173
Intercepto: 74.628
  horas_redes_sociais                 coef = -5.1704 | beta padronizado = -0.6328


> 🔎 **Interpretação.**
> - **R² = 0,402:** as horas em redes sozinhas explicam cerca de **40%** da variação do bem-estar.
> - **Coeficiente = -5,17:** cada hora a mais em redes se associa a **5,2 pontos a menos** de bem-estar.
> - **MAE = 6,5:** em média, a previsão erra cerca de 6,5 pontos de bem-estar.
> - No gráfico, a reta acompanha bem a nuvem de pontos, o que indica que a relação é bem descrita por uma linha.

#### Modelo B — Bem-estar ~ redes + sono

In [16]:
resultados.append(ajustar("B. Bem-estar ~ redes + sono", "indice_bem_estar", ["horas_redes_sociais", "horas_sono"]))


B. Bem-estar ~ redes + sono
n = 500,000 | R² (teste) = 0.5107 | MAE = 5.902 | RMSE = 7.394
Intercepto: 30.457
  horas_redes_sociais                 coef = -1.9749 | beta padronizado = -0.2426
  horas_sono                          coef = +5.5803 | beta padronizado = +0.5123
Correlação entre preditores (checar colinearidade):


horas_redes_sociais,horas_sono
f64,f64
1.0,-0.761701
-0.761701,1.0


> 🔎 **Interpretação.**
> - **R² = 0,511:** incluir o sono eleva a explicação de 40% para **51%**, e o erro cai (MAE de 6,5 para 5,9).
> - **Peso de cada fator:** o beta do sono (**+0,51**) é mais que o dobro do beta das redes (**-0,24**). Com os dois no modelo, o **sono é o fator mais associado ao bem-estar**.
> - **O efeito das redes encolhe:** o coeficiente cai de **-5,17** (modelo A) para **-1,97**, uma redução de cerca de 62%. Mantendo o sono constante, cada hora em redes se associa a 2,0 pontos a menos de bem-estar, e cada hora de sono se associa a 5,6 pontos a mais.
> - **Colinearidade:** a correlação entre redes e sono é de **-0,76** (fator de inflação da variância de cerca de 2,4, moderado). Os coeficientes continuam interpretáveis, mas os dois efeitos não são totalmente separáveis.

#### Modelo C — Sono ~ redes sociais

In [15]:
resultados.append(ajustar("C. Sono ~ redes", "horas_sono", ["horas_redes_sociais"]))


C. Sono ~ redes
n = 500,000 | R² (teste) = 0.5801 | MAE = 0.504 | RMSE = 0.629
Intercepto: 7.916
  horas_redes_sociais                 coef = -0.5727 | beta padronizado = -0.7617


> 🔎 **Interpretação.**
> - **R² = 0,580:** é o melhor ajuste do estudo. As redes explicam cerca de **58%** da variação do sono.
> - **Coeficiente = -0,57:** cada hora em redes se associa a cerca de **34 minutos a menos de sono**.
> - **MAE = 0,50 h** (cerca de 30 minutos): erro médio pequeno.
> - **Piso de 4 h:** os gráficos mostram um acúmulo de estudantes em 4,0 h de sono. A reta prevê valores abaixo de 4 h para uso muito alto (por exemplo, cerca de 3,3 h com 8 h de redes), o que não ocorre nos dados.

### 4.2 Modelos de desempenho
Frequência às aulas e capacidade de atenção como variáveis resposta.

#### Modelo D — Frequência ~ redes + sono

In [14]:
resultados.append(ajustar("D. Frequência ~ redes + sono", "taxa_frequencia_aulas", ["horas_redes_sociais", "horas_sono"]))


D. Frequência ~ redes + sono
n = 500,000 | R² (teste) = 0.0791 | MAE = 4.795 | RMSE = 5.874
Intercepto: 91.400
  horas_redes_sociais                 coef = -1.0313 | beta padronizado = -0.2186
  horas_sono                          coef = +0.4750 | beta padronizado = +0.0761
Correlação entre preditores (checar colinearidade):


horas_redes_sociais,horas_sono
f64,f64
1.0,-0.761701
-0.761701,1.0


> 🔎 **Interpretação.**
> - **R² = 0,079:** os dois fatores explicam menos de **8%** da variação da frequência. Outros fatores pesam bem mais.
> - **Peso de cada fator:** as redes (beta **-0,22**) pesam quase **três vezes** mais que o sono (beta **+0,08**).
> - **Efeito:** cada hora em redes se associa a cerca de **1 ponto percentual a menos** de frequência.
> - **Teto de 100%:** no gráfico real × previsto, muitos estudantes se acumulam em 100%, e o modelo prevê valores concentrados entre cerca de 85% e 95%.

#### Modelo E — Atenção ~ redes + sono

In [13]:
resultados.append(ajustar("E. Atenção ~ redes + sono", "capacidade_atencao_minutos", ["horas_redes_sociais", "horas_sono"]))


E. Atenção ~ redes + sono
n = 500,000 | R² (teste) = 0.1972 | MAE = 3.672 | RMSE = 4.506
Intercepto: 54.333
  horas_redes_sociais                 coef = -1.3097 | beta padronizado = -0.3412
  horas_sono                          coef = +0.6300 | beta padronizado = +0.1210
Correlação entre preditores (checar colinearidade):


horas_redes_sociais,horas_sono
f64,f64
1.0,-0.761701
-0.761701,1.0


> 🔎 **Interpretação.**
> - **R² = 0,197:** os dois fatores explicam cerca de **20%** da variação da atenção.
> - **Peso de cada fator:** as redes (beta **-0,34**) pesam quase **três vezes** mais que o sono (beta **+0,12**).
> - **Efeito:** cada hora em redes se associa a cerca de **1,3 minuto a menos** de capacidade de atenção, e o erro médio é de 3,7 minutos.
> - **Teto de 60 min:** como no modelo D, há acúmulo de estudantes no valor máximo (60 minutos).

### 4.3 Comparação dos modelos
Resumo das métricas de todos os modelos, ordenado pelo R² no teste.

> ▶️ Execute a célula (após rodar os cinco modelos) para gerar a tabela e o gráfico de barras. A tabela da sequência já traz os valores completos.

In [ ]:
resumo = pl.DataFrame(resultados).sort("r2_teste", descending=True)
display(resumo.select("modelo", "r2_teste", "mae", "rmse"))

plt.figure(figsize=(8, 3.5))
sns.barplot(data=resumo.to_pandas(), y="modelo", x="r2_teste", color="steelblue")
plt.title("R² no conjunto de teste")
plt.xlabel("R²")
plt.ylabel("")
plt.tight_layout()
plt.show()

**Resumo completo dos modelos**

| Posição | Modelo | R² (teste) | MAE | RMSE |
|---|---|---|---|---|
| 1 | **C.** Sono ~ redes | 0,5801 | 0,504 h | 0,629 h |
| 2 | **B.** Bem-estar ~ redes + sono | 0,5107 | 5,902 | 7,394 |
| 3 | **A.** Bem-estar ~ redes | 0,4021 | 6,535 | 8,173 |
| 4 | **E.** Atenção ~ redes + sono | 0,1972 | 3,672 min | 4,506 min |
| 5 | **D.** Frequência ~ redes + sono | 0,0791 | 4,795 p.p. | 5,874 p.p. |

**Peso das variáveis (betas padronizados)**

| Modelo | Redes | Sono |
|---|---|---|
| A. Bem-estar | -0,633 | — |
| B. Bem-estar | -0,243 | +0,512 |
| C. Sono | -0,762 | — |
| D. Frequência | -0,219 | +0,076 |
| E. Atenção | -0,341 | +0,121 |

> 🔎 **Interpretação.**
> - Os modelos de **saúde** (C, B e A) têm os maiores R², e os de **desempenho** (E e D) têm os menores, o mesmo gradiente visto nas correlações.
> - Os R² dos modelos simples coincidem com o **r² das correlações** (por exemplo, 0,58 para o sono e 0,40 para o bem-estar), o que confirma a consistência entre as duas etapas.
> - **O sono quase não acrescenta explicação ao desempenho:** o R² da frequência passa de 0,077 (só redes, correlação) para 0,079, e o da atenção de 0,188 para 0,197.
> - **Pesos opostos:** no **bem-estar**, o sono pesa mais que as redes. Na **atenção e na frequência**, as redes pesam mais que o sono.

---
## 5. Discussão

**O papel do sono**

- Ao incluir o sono, o efeito das redes sobre o bem-estar cai cerca de 62%. Isso é **compatível** com a ideia de que parte do efeito das redes passa pela redução do sono, mas **não a comprova**: como redes e sono são muito correlacionados (r = -0,76), a divisão entre os dois efeitos também depende dessa sobreposição.
- Para atenção e frequência, o efeito das redes permanece mesmo com o sono no modelo, o que indica uma associação que **não depende apenas do sono**.

**Qualidade dos modelos**

- Nos modelos A e B, a reta (ou a diagonal do gráfico real × previsto) acompanha bem a nuvem de pontos, o que indica bom ajuste linear.
- Nos modelos de frequência e de atenção, há **tetos** (100% e 60 minutos) visíveis no gráfico real × previsto. A regressão linear **não respeita esses limites**, então suas previsões são menos confiáveis, sobretudo para estudantes perto do máximo.

**Como ler os valores de R²**

- Um R² baixo, como o de 0,079 na frequência, é um resultado válido: significa que redes e sono explicam pouco e que outros fatores pesam mais.
- Um R² alto não indica causa: indica que a variável explica bem a variação, em uma relação observada.

---
## 6. Conclusões

**Conclusões**

1. As horas em redes sociais explicam **58% do sono** e **40% do bem-estar**: cada hora a mais se associa a cerca de **34 minutos a menos de sono** e a **5,2 pontos a menos de bem-estar**.
2. Ao incluir o sono, a explicação do bem-estar sobe para **51%**, e o **sono passa a ser o fator de maior peso**.
3. Para o **desempenho**, os modelos explicam pouco: **20% da atenção** e **8% da frequência**, com as redes pesando mais que o sono.
4. O padrão é consistente com a Parte 1 e com as correlações: o efeito é mais forte na **saúde** do que no **desempenho**.

